# VisionX P4 - data prep + training on Colab
Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom.

In [ ]:
!nvidia-smi | head -15

In [ ]:
# Upload visionx_p4.zip (the project zip), or git clone your repo instead
from google.colab import files
up = files.upload()
!unzip -oq visionx_p4.zip -d /content
%cd /content/visionx_p4
# !git clone https://github.com/<you>/<repo>.git /content/visionx_p4 && %cd /content/visionx_p4

In [ ]:
# NYU loader on HF is a loading script -> needs datasets<4
!pip install -q "datasets<4" h5py opencv-python-headless scikit-image psutil

In [ ]:
# RAM-heavy step: stream NYU Depth V2, resize to 240x320, save arrays (~10-20 min)
!python prepare_data.py --source hf --out /content/data/nyu240 --n_train 2000 --n_val 200 --stride 10

# Fallback (official labeled .mat, 1449 imgs, ~2.8 GB):
# !wget -q http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat
# !python prepare_data.py --source mat --mat_path nyu_depth_v2_labeled.mat --out /content/data/nyu240 --n_val 249

In [ ]:
# Train (~1-2 h on a T4 for 15 epochs; lower --epochs if short on time)
!python train.py --data /content/data/nyu240 --epochs 15 --bs 8 --out weights/hybrid_unet.pt

In [ ]:
# Comparative evaluation + ablation + failure cases -> eval_P4_T39.md
!python evaluate.py --data /content/data/nyu240 --n 100 --usns "USN1,USN2,USN3,USN4"

In [ ]:
# Download results for the local machine
!zip -j val_data.zip /content/data/nyu240/val_*.npy
from google.colab import files
files.download('weights/hybrid_unet.pt')
files.download('weights/train_log.csv')
files.download('eval_P4_T39.md')
files.download('val_data.zip')
!zip -qr failure_cases.zip outputs/failure_cases && echo zipped
files.download('failure_cases.zip')